In [3]:
! pip -qqq install transformers datasets trl torch
! pip -qqq install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.9 MB/s eta 0:00:00


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [6]:
dataset_name = 'HuggingFaceTB/smoltalk2'
ds = load_dataset(dataset_name, 'SFT', streaming=True)

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

In [7]:
ds.keys()

dict_keys(['LongAlign_64k_Qwen3_32B_yarn_131k_think', 'OpenThoughts3_1.2M_think', 'aya_dataset_Qwen3_32B_think', 'multi_turn_reasoning_if_think', 's1k_1.1_think', 'smolagents_toolcalling_traces_think', 'smoltalk_everyday_convs_reasoning_Qwen3_32B_think', 'smoltalk_multilingual8_Qwen3_32B_think', 'smoltalk_systemchats_Qwen3_32B_think', 'table_gpt_Qwen3_32B_think', 'LongAlign_64k_context_lang_annotated_lang_6_no_think', 'Mixture_of_Thoughts_science_no_think', 'OpenHermes_2.5_no_think', 'OpenThoughts3_1.2M_no_think_no_think', 'hermes_function_calling_v1_no_think', 'smoltalk_multilingual_8languages_lang_5_no_think', 'smoltalk_smollm3_everyday_conversations_no_think', 'smoltalk_smollm3_explore_instruct_rewriting_no_think', 'smoltalk_smollm3_smol_magpie_ultra_no_think', 'smoltalk_smollm3_smol_rewrite_no_think', 'smoltalk_smollm3_smol_summarize_no_think', 'smoltalk_smollm3_systemchats_30k_no_think', 'table_gpt_no_think', 'tulu_3_sft_personas_instruction_following_no_think', 'xlam_traces_no_th

In [8]:
!find ~/.cache/huggingface/hub -iname "*SmolLM-135M*" -type d

In [9]:
!cat ~/.cache/huggingface/hub/models--HuggingFaceTB--SmolLM-135M/snapshots/*/config.json

cat: '/root/.cache/huggingface/hub/models--HuggingFaceTB--SmolLM-135M/snapshots/*/config.json': No such file or directory


In [10]:
split = 'Mixture_of_Thoughts_science_no_think'

In [11]:
model_name = "HuggingFaceTB/SmolLM-135M"
base = AutoModelForCausalLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name + '-Instruct')

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/3.59k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/565 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

In [12]:
instucut_name = model_name + '-Instruct'
instruct = AutoModelForCausalLM.from_pretrained(instucut_name).to(device)


model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

In [13]:
print(f"Model Size: {(base.get_memory_footprint() / 1024**3):.3f}GB")

Model Size: 0.251GB


In [14]:
from datasets import Dataset
examples_list = list(ds[split].take(5))
examples = Dataset.from_list(examples_list)

In [15]:
tokenizer.chat_template

"{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"

In [16]:
def format_ds(batch):
    return {'text': [tokenizer.apply_chat_template(ex, tokenize=False) for ex in batch['messages']]}

text = examples.map(format_ds, batched=True, remove_columns=examples.column_names)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [17]:
text[0]

{'text': "<|im_start|>user\nWhat hormone's action is inhibited by caffeine, leading to increased urination?A: ADH\nB: Insulin\nC: Thyroxine\nD: Cortisol<|im_end|>\n<|im_start|>assistant\nCaffeine acts as a diuretic by inhibiting the action of antidiuretic hormone (ADH), which is responsible for signaling the kidneys to reabsorb water and concentrate urine. When ADH is suppressed, the kidneys excrete more water, leading to increased urination. The other hormones listed—insulin (regulates blood sugar), thyroxine (regulates metabolism), and cortisol (involved in stress response)—are not directly related to fluid balance or diuresis. \n\n**Answer: A**  \n\\boxed{A}<|im_end|>\n"}

In [18]:
tokenizer.eos_token_id

2

In [19]:
# prompt = "I'd like a recipe for something warm."
# tokenized = tokenizer(prompt, return_tensors='pt').to(device)

# with torch.no_grad():
#   outputs = base.generate(
#       **tokenized,
#       max_new_tokens=200,
#       temperature=0.7,
#       do_sample=True,
#       pad_token_id=tokenizer.eos_token_id
#   )
#   decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
#   print(decoded)



In [20]:
torch.cuda.is_available()

True

In [21]:
base.device

device(type='cuda', index=0)

In [22]:
instruct.name_or_path

'HuggingFaceTB/SmolLM-135M-Instruct'

In [23]:
# # Sampling from the instruct model

# message = [{
#     "role": "user",
#     "content": prompt
# }]

# formatted = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)

# formatted_tokenized = tokenizer(formatted, return_tensors='pt').to(device)

# output = instruct.generate(
#     **formatted_tokenized,
#     max_new_tokens=200,
#     temperature=0.5,
#     do_sample=True,
#     pad_token_id=tokenizer.eos_token_id
# )

# decoded = tokenizer.decode(output[0])
# print(decoded)
# #

In [ ]:
# assistant_start = tokenizer.bos_token + 'assistant\n'
# response_start = decoded.find(assistant_start) + len(assistant_start)
# response = decoded[response_start:].split(tokenizer.eos_token)[0]
# print(response)

### Now FT the base model

In [25]:
def preprocess(batch):
  return {'text': [tokenizer.apply_chat_template(ex, tokenize=False) for ex in batch['messages']]}


In [26]:
training_config = SFTConfig(
    # Model and data
    output_dir=f"./{model_name}",
    dataset_text_field="text",
    max_length=512,

    # Training hyperparameters
    per_device_train_batch_size=160,  # Tried mutliple batchsizes for a few iters. 160 batch size uses 12.8/15 GB.
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    num_train_epochs=1,  # Start with 1 epoch
    # Need to estimate how many tokens the splits i'm using have to pick a good iter number.
    # Considering sequence packing, it's hard to get an accurate computation of how many batches the dataset fits.
    max_steps=2000,

    # Optimization
    warmup_steps=50,
    weight_decay=0.01,
    optim="adamw_torch",

    # Logging and saving
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    save_total_limit=2,

    # Memory optimization
    dataloader_num_workers=0,
    # group_by_length=True,  # Group similar length sequences

    # Hugging Face Hub integration
    push_to_hub=False,  # Set to True to upload to Hub
    hub_model_id=f"your-username/{model_name}",

    # Experiment tracking
    # report_to=["trackio"],  # Use trackio for experiment tracking
    run_name=f"{model_name}-training",
)

print("Training configuration set!")
print(f"Effective batch size: {training_config.per_device_train_batch_size * training_config.gradient_accumulation_steps}")

Training configuration set!
Effective batch size: 320


In [38]:
used_splits = [
    'smoltalk_smollm3_everyday_conversations_no_think','Mixture_of_Thoughts_science_no_think',
    'tulu_3_sft_personas_instruction_following_no_think',
    'smoltalk_multilingual_8languages_lang_5_no_think',
    'table_gpt_no_think'
    ]

# Print a row from each split to check format.
for split in used_splits:
    example = next(iter(ds[split].take(1)))
    print(f"--- {split} ---")
    print(example)
    print()


--- smoltalk_smollm3_everyday_conversations_no_think ---
{'messages': [{'content': 'Hi there', 'role': 'user'}, {'content': 'Hello! How can I help you today?', 'role': 'assistant'}, {'content': "I'm looking for a healthy breakfast idea. What's a good option?", 'role': 'user'}, {'content': "A fruit salad is a great choice. It's nutritious, delicious, and easy to make.", 'role': 'assistant'}, {'content': 'What fruits are good in a fruit salad?', 'role': 'user'}, {'content': 'You can use a mix of your favorite fruits, such as strawberries, bananas, grapes, and pineapple.', 'role': 'assistant'}], 'chat_template_kwargs': {'custom_instructions': '', 'enable_thinking': False, 'python_tools': [], 'xml_tools': []}, 'source': 'smoltalk-smollm3_everyday-conversations'}

--- Mixture_of_Thoughts_science_no_think ---
{'messages': [{'content': "What hormone's action is inhibited by caffeine, leading to increased urination?A: ADH\nB: Insulin\nC: Thyroxine\nD: Cortisol", 'role': 'user'}, {'content': 'C

In [42]:
from datasets import interleave_datasets
eval_ex_num = 400
eval_ds = []
train_ds = []
for split in used_splits:
  pass

In [28]:
from datasets import load_dataset_builder
row_count = 0
builder = load_dataset_builder("HuggingFaceTB/smoltalk2", "SFT")
for split in used_splits:
  count = builder.info.splits[split].num_examples
  row_count += count
  print(f"{split}: {count}")

print(f"Total rows: {row_count}")

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

smoltalk_smollm3_everyday_conversations_no_think: 2260
Mixture_of_Thoughts_science_no_think: 86110
tulu_3_sft_personas_instruction_following_no_think: 29970
smoltalk_multilingual_8languages_lang_5_no_think: 254047
table_gpt_no_think: 13203
Total rows: 385590


In [47]:
int(row_count*0.10)

38559

In [56]:
x = next(iter(ds[used_splits[0]]))
x

{'messages': [{'content': 'Hi there', 'role': 'user'},
  {'content': 'Hello! How can I help you today?', 'role': 'assistant'},
  {'content': "I'm looking for a healthy breakfast idea. What's a good option?",
   'role': 'user'},
  {'content': "A fruit salad is a great choice. It's nutritious, delicious, and easy to make.",
   'role': 'assistant'},
  {'content': 'What fruits are good in a fruit salad?', 'role': 'user'},
  {'content': 'You can use a mix of your favorite fruits, such as strawberries, bananas, grapes, and pineapple.',
   'role': 'assistant'}],
 'chat_template_kwargs': {'custom_instructions': '',
  'enable_thinking': False,
  'python_tools': [],
  'xml_tools': []},
 'source': 'smoltalk-smollm3_everyday-conversations'}

In [62]:
# Checking how many big one sequence is.
# Does TRL handle if a tokenized row produces more tokens than sequence_len?
tokenized_example = tokenizer.apply_chat_template(x['messages'], tokenize=True)
tokenized_example['input_ids'].__len__()

108

In [46]:
! pip show trl

Name: trl
Version: 1.9.2
Summary: Train transformer language models with reinforcement learning.
Home-page: https://github.com/huggingface/trl
Author: 
Author-email: Leandro von Werra <leandro.vonwerra@gmail.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: accelerate, datasets, jinja2, packaging, transformers
Required-by: 


In [ ]:
# trainer = SFTTrainer(
#     model=base,
#     train_dataset=selected_ds,
#     args=training_config,
#     processing_class=tokenizer,
# )

In [ ]:
# trainer.train()